# 02 · Prepare the model inputs
Use the 21 named predictors from `src/preprocessing.py`. These describe maternal and household background, pregnancy history and experiences, and delivery mode. Identical predictor profiles stay in the same split. The remaining fields include serial numbers, questionnaire scores and outcomes, and measures of current postpartum mood or functioning.

Median imputation and scaling are used for numeric features. Most-frequent imputation and one-hot encoding are used for categories. Fit these operations inside the model pipeline in every training fold.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import pandas as pd
from IPython.display import display
from src.preprocessing import FEATURES, LABELS, TARGET, export_processed_data, make_split, profile_groups, read_data

X_train, X_test, y_train, y_test = make_split()
print(f"Training: {X_train.shape} | Held-out test: {X_test.shape}")
display(pd.DataFrame({"train": y_train.value_counts(), "test": y_test.value_counts()})
        .sort_index().rename(index=dict(enumerate(LABELS))))
assert set(X_train.index).isdisjoint(X_test.index)
assert set(profile_groups(X_train)).isdisjoint(profile_groups(X_test))
assert not {"sr", TARGET, "EPDS Score", "PHQ9 Score", "PHQ9 Result"} & set(FEATURES)
print("Excluded:", ", ".join(c for c in read_data().columns if c not in FEATURES))
print("Predictors:", ", ".join(FEATURES))

Training: (640, 21) | Held-out test: (160, 21)


,train,test
EPDS Result,,
Low,207,52
Medium,152,38
High,281,70


Excluded: sr, Occupation After Your Latest Childbirth, Current monthly income, Husband’s monthly income, Addiction, Total children, Relationship with the in-laws, Relationship with husband, Relationship with the newborn, Relationship between father and newborn, Feeling about motherhood, Recieved Support, Need for Support, Abuse, Trust and share feelings, Age of newborn, Age of immediate older children, Gender of newborn, Birth compliancy, Breastfeed, Newborn illness, Worry about newborn, Relax/sleep when newborn is tended, Relax/sleep when the newborn is asleep, Angry after latest child birth, Feeling for regular activities, PHQ9 Score, PHQ9 Result, EPDS Score, EPDS Result
Predictors: Age, Number of the latest pregnancy, Residence, Education Level, Marital status, Occupation before latest pregnancy, Monthly income before latest pregnancy, Husband's education level, Disease before pregnancy, History of pregnancy loss, Family type, Number of household members, Major changes or losses dur

In [2]:
# Fit one reference preprocessor on training data and export both splits.
encoded_train = export_processed_data(X_train, X_test, y_train, y_test)
print("Missing selected predictors in training:", X_train.isna().sum().sum())
print("Training matrix after encoding:", encoded_train.shape)
print("First encoded columns:", encoded_train.columns[:12].tolist())

Missing selected predictors in training: 1
Training matrix after encoding: (640, 63)
First encoded columns: ['Age', 'Number of the latest pregnancy', 'Residence_city', 'Residence_village', 'Education Level_college', 'Education Level_high school', 'Education Level_none', 'Education Level_primary school', 'Education Level_university', 'Marital status_divorced', 'Marital status_married', 'Occupation before latest pregnancy_business']


The held-out split is reproduced with a fixed seed. The reference encoder exports a shared feature matrix for surrogate work. The calibrated model accepts the cleaned 21-column inputs. Later notebooks create fresh pipelines so each cross-validation fold learns its own imputation, scaling and encoding.